# 02. California DWR Bulletin 120 Forecast Archive Download

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/02_download_b120_forecasts.ipynb)

This notebook downloads historical operational **California DWR Bulletin 120**
April–July unimpaired runoff forecasts from CDEC across all 23 Bulletin 120
California snow-fed watersheds and consolidates them into Zarr:
- **Output path**: `${FLOWRACLE_DATA_DIR}/b120-forecasts/b120_consolidated.zarr`
- **CDEC Sensors**:
  - Sensor `262`: 90% exceedance forecast $\mapsto$ quantile $q = 0.10$
  - Sensor `261`: 50% exceedance (median) forecast $\mapsto$ quantile $q = 0.50$
  - Sensor `260`: 10% exceedance forecast $\mapsto$ quantile $q = 0.90$

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0"

In [ ]:
from datetime import datetime
import io
import os
import shutil
import time

import pandas as pd
import requests
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
DATASET_DIR = os.path.join(FLOWRACLE_DATA_DIR, "b120-forecasts")
RAW_CSV_DIR = os.path.join(DATASET_DIR, "raw_csv")
os.makedirs(RAW_CSV_DIR, exist_ok=True)

VERIFICATION_MODE = False
DEFAULT_START_YEAR = 2005
END_YEAR = datetime.now().year
STATION_START_YEARS = {
    "FTO": 2000,
    "SIS": 2000,
    "MSS": 2000,
    "SDT": 2005,
    "PSH": 2005,
    "KWT": 2005,
    "SBB": 2005,
}
SENSORS = [261] if VERIFICATION_MODE else [261, 260, 262]
START_YEAR = END_YEAR - 1 if VERIFICATION_MODE else DEFAULT_START_YEAR

B120_23_STATIONS = [
    "AMF",
    "CSN",
    "EFC",
    "EWR",
    "FTO",
    "KGF",
    "KRI",
    "KWT",
    "MKM",
    "MRC",
    "MSS",
    "PSH",
    "SBB",
    "SBF",
    "SCC",
    "SDT",
    "SIS",
    "SNS",
    "TLG",
    "TRF",
    "WFC",
    "WWR",
    "YRS",
]

## 1. Query Seasonal Forecast Tables from CDEC

In [ ]:
def download_b120(
    stations,
    start_year_default,
    end_year,
    sensors,
    station_start_years=None,
    out_root=RAW_CSV_DIR,
):
  base_url = "https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet"
  target_stations = stations[:3] if VERIFICATION_MODE else stations

  for sensor in sensors:
    sensor_dir = os.path.join(out_root, str(sensor))
    os.makedirs(sensor_dir, exist_ok=True)
    for station in target_stations:
      s_year = (
          station_start_years.get(station, start_year_default)
          if (not VERIFICATION_MODE and station_start_years)
          else start_year_default
      )
      frames = []
      for year in range(s_year, end_year + 1):
        params = {
            "Stations": station,
            "SensorNums": sensor,
            "dur_code": "E",
            "Start": f"{year}-01-01",
            "End": f"{year}-06-01",
        }
        for attempt in range(5):
          try:
            resp = requests.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text))
            if not df.empty and "VALUE" in df.columns:
              frames.append(df)
            break
          except (requests.RequestException, ConnectionResetError):
            if attempt < 4:
              time.sleep(3 * (2**attempt))
      if frames:
        out_csv = os.path.join(sensor_dir, f"{station}.csv")
        pd.concat(frames, ignore_index=True).to_csv(out_csv, index=False)
        print(
            f"[Sensor {sensor}] Saved {station} ({sum(len(x) for x in frames)}"
            " rows)"
        )


download_b120(
    B120_23_STATIONS, START_YEAR, END_YEAR, SENSORS, STATION_START_YEARS
)

## 2. Consolidate Exceedance Sensors into Quantile Zarr Store

In [ ]:
def consolidate_b120_to_zarr(raw_root, zarr_out_path):
  # Note: 90% exceedance corresponds to the 0.10 runoff quantile; 10% exceedance to 0.90.
  sensor_to_quantile = {262: 0.1, 261: 0.5, 260: 0.9}
  sensor_datasets = []

  for sensor, q_val in sensor_to_quantile.items():
    sensor_dir = os.path.join(raw_root, str(sensor))
    if not os.path.isdir(sensor_dir):
      continue
    station_datasets = []
    for fname in sorted(os.listdir(sensor_dir)):
      if not fname.endswith(".csv"):
        continue
      station = os.path.splitext(fname)[0]
      df = pd.read_csv(
          os.path.join(sensor_dir, fname),
          parse_dates=["DATE TIME", "OBS DATE"],
          low_memory=False,
      )
      if df.empty or "DATE TIME" not in df.columns or "VALUE" not in df.columns:
        continue
      cols_exist = [
          c
          for c in ["DATE TIME", "OBS DATE", "VALUE", "DATA_FLAG"]
          if c in df.columns
      ]
      df = df[cols_exist].copy()
      df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
      df = df.rename(
          columns={
              "DATE TIME": "time",
              "OBS DATE": "obs_date",
              "VALUE": "forecast_fnf_april_july",
              "DATA_FLAG": "data_flag",
          }
      )
      df["station_id"] = station
      df["quantile"] = q_val
      df = df[
          ~df.duplicated(subset=["time", "station_id", "quantile"], keep="last")
      ]
      station_datasets.append(
          df.set_index(["time", "station_id", "quantile"]).to_xarray()
      )
    if station_datasets:
      sensor_datasets.append(
          xr.concat(station_datasets, dim="station_id", join="outer")
      )

  combined = xr.concat(sensor_datasets, dim="quantile", join="outer").sortby(
      ["station_id", "time", "quantile"]
  )
  combined["forecast_fnf_april_july"].attrs = {
      "description": (
          "Forecasted Unimpaired Runoff Volume for April-July period."
      ),
      "units": "AF",
  }
  combined.attrs = {
      "description": "Consolidated B120 Forecast Dataset.",
      "origin": "California Department of Water Resources (DWR) via CDEC",
  }
  if os.path.exists(zarr_out_path):
    shutil.rmtree(zarr_out_path)
  combined.to_zarr(zarr_out_path, mode="w")
  print(f"Wrote Zarr store: {zarr_out_path}")
  return combined


b120_zarr_path = os.path.join(DATASET_DIR, "b120_consolidated.zarr")
b120_ds = consolidate_b120_to_zarr(RAW_CSV_DIR, b120_zarr_path)
print(b120_ds)